In [1]:
# ── BLOCK 1 : Load ───────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import joblib

X_train = pd.read_csv('../data/processed/X_train.csv')
X_test  = pd.read_csv('../data/processed/X_test.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()
y_test  = pd.read_csv('../data/processed/y_test.csv').squeeze()

# also load the full df — needed to add dashboard columns
df = pd.read_csv('../data/processed/df_preprocessed.csv')

print("X_train:", X_train.shape)
print("X_test: ", X_test.shape)

X_train: (400, 36)
X_test:  (100, 36)


In [2]:
# ── BLOCK 2 : Customer Health Score ─────────────────────────────────────────
# Formula: high usage + high satisfaction + low errors + low escalations = healthy
# Each component normalised 0-1, then combined into 0-100 score

def compute_health_score(df_input):
    d = df_input.copy()

    # normalise each component to 0-1 range
    def minmax(series):
        mn, mx = series.min(), series.max()
        if mx == mn:
            return pd.Series(0.5, index=series.index)
        return (series - mn) / (mx - mn)

    usage_score       = minmax(d['total_usage_count'])          # high = good
    satisfaction_score = minmax(d['mean_satisfaction'])          # high = good
    error_score       = 1 - minmax(d['total_errors'])           # low errors = good
    escalation_score  = 1 - minmax(d['escalation_count'])       # low escalations = good
    feature_score     = minmax(d['distinct_features'])          # breadth = good

    # weighted average — tune weights based on domain knowledge
    health = (
        0.25 * usage_score        +
        0.25 * satisfaction_score +
        0.20 * error_score        +
        0.15 * escalation_score   +
        0.15 * feature_score
    ) * 100

    return health.round(1)

# apply to train, test, and full df
X_train['health_score'] = compute_health_score(X_train)
X_test['health_score']  = compute_health_score(X_test)
df['health_score']      = compute_health_score(df)

print("Health score stats:")
print(X_train['health_score'].describe().round(2))

Health score stats:
count    400.00
mean      56.85
std        8.36
min       36.70
25%       51.28
50%       56.95
75%       62.20
max       78.80
Name: health_score, dtype: float64


In [3]:
# ── BLOCK 3 : Rule-based Risk Score ─────────────────────────────────────────
# Each condition is a known churn risk factor from EDA
# Score range: 0 (no risk factors) to 100 (all risk factors present)

def compute_risk_score(df_input):
    score = pd.Series(0, index=df_input.index, dtype=float)

    # low usage → +25
    if 'is_low_usage' in df_input.columns:
        score += df_input['is_low_usage'] * 25

    # auto renew off → +20
    if 'auto_renew' in df_input.columns:
        score += (df_input['auto_renew'] == 0).astype(int) * 20

    # preceding downgrade → +20
    if 'preceding_downgrade' in df_input.columns:
        score += df_input['preceding_downgrade'] * 20

    # ever downgraded → +15
    if 'ever_downgraded' in df_input.columns:
        score += df_input['ever_downgraded'] * 15

    # high error rate → +10  (above median)
    if 'error_rate' in df_input.columns:
        median_err = df_input['error_rate'].median()
        score += (df_input['error_rate'] > median_err).astype(int) * 10

    # low satisfaction → +10
    if 'mean_satisfaction' in df_input.columns:
        score += (df_input['mean_satisfaction'] < 3).astype(int) * 10

    return score.clip(0, 100).round(1)

X_train['risk_score'] = compute_risk_score(X_train)
X_test['risk_score']  = compute_risk_score(X_test)
df['risk_score']      = compute_risk_score(df)

print("Risk score distribution:")
print(X_train['risk_score'].value_counts().sort_index())

Risk score distribution:
risk_score
10.0    82
20.0    78
25.0    50
30.0     9
35.0    90
40.0     7
45.0    45
50.0    11
55.0    10
60.0    12
65.0     3
70.0     3
Name: count, dtype: int64


In [4]:
# ── BLOCK 4 : Engagement Segment (dashboard label) ──────────────────────────

def assign_segment(df_input):
    conditions = [
        (df_input['health_score'] >= 70) & (df_input['total_usage_count'] > df_input['total_usage_count'].median()),
        (df_input['health_score'] >= 50),
        (df_input['health_score'] >= 30),
    ]
    choices = ['Champion', 'Healthy', 'At Risk']
    return np.select(conditions, choices, default='Dormant')

X_train['engagement_segment'] = assign_segment(X_train)
X_test['engagement_segment']  = assign_segment(X_test)
df['engagement_segment']      = assign_segment(df)

print("Segment distribution (train):")
print(X_train['engagement_segment'].value_counts())

# encode segment for model use
from sklearn.preprocessing import LabelEncoder
seg_encoder = LabelEncoder()
X_train['engagement_segment'] = seg_encoder.fit_transform(X_train['engagement_segment'])
X_test['engagement_segment']  = seg_encoder.transform(X_test['engagement_segment'])
# keep string version in df for dashboard
# df['engagement_segment'] stays as string — do NOT encode df

Segment distribution (train):
engagement_segment
Healthy     299
At Risk      78
Champion     23
Name: count, dtype: int64


In [5]:
# ── BLOCK 5 : Verify + Save ──────────────────────────────────────────────────

print("New features added:")
new_cols = ['health_score', 'risk_score', 'engagement_segment']
print(X_train[new_cols].describe().round(2))

print("\nX_train shape:", X_train.shape)
print("X_test shape: ", X_test.shape)
print("Any nulls:", X_train.isnull().sum().sum() + X_test.isnull().sum().sum())

# save updated train/test
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv',  index=False)

# save full df with health_score, risk_score, segment (for dashboard)
df.to_csv('../data/processed/df_preprocessed.csv', index=False)

# save segment encoder for Streamlit
import joblib
joblib.dump(seg_encoder, '../models/segment_encoder.pkl')

print("\nSaved X_train, X_test, df_preprocessed, segment_encoder.pkl")

New features added:
       health_score  risk_score  engagement_segment
count        400.00      400.00              400.00
mean          56.85       28.95                1.55
std            8.36       14.54                0.80
min           36.70       10.00                0.00
25%           51.28       20.00                1.00
50%           56.95       25.00                2.00
75%           62.20       35.00                2.00
max           78.80       70.00                2.00

X_train shape: (400, 39)
X_test shape:  (100, 39)
Any nulls: 0

Saved X_train, X_test, df_preprocessed, segment_encoder.pkl
